# Surface reconstruction

Choose the folder containing the reconstructed LAS/LAZ files and the filename suffix
to process. Matching files are read from that folder and its subfolders.
Choose one surface type and edit its parameters, then run the notebook from top to bottom.

- **Voxel Grid Downsampling** (`voxel`): divide XYZ into cubes and replace the points
  in each occupied cube with their centroid. Set the cube edge length with `size`.
- **XY Grid-Based Surface Extraction** (`xy_grid`): divide XY into cells and output
  one point at each cell's mean XY. Choose mean, median, minimum, maximum or percentile
  Z with `statistic`. Set `size`, `minimum_points` per cell and, when applicable, `percentile`.
- **Moving Least Squares Surface Smoothing** (`mls`): fit a weighted local polynomial
  and update Z at each input XY. Set `degree` to 1 for a plane or 2 for a quadratic.
- **Local Surface Fitting and Point Projection** (`local_plane`): fit a weighted local
  plane in XYZ and project each input point onto it.

For MLS and local planes, `neighbors` caps the neighbourhood, `radius` limits its
extent (`None` removes the limit), `bandwidth` controls Gaussian weights, and
`minimum_neighbors` sets the minimum support. Points with insufficient or degenerate
fits retain their input position. Distances use the input coordinate units.


In [39]:
from datetime import datetime, timezone
from pathlib import Path
import sys
import uuid

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / 'LS_Slopes_reconstruction_Multiple_iterations' / 'surface_extraction.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Start the notebook inside the project folder.')
SURFACE_DIR = PROJECT_ROOT / 'LS_Slopes_reconstruction_Multiple_iterations'
if str(SURFACE_DIR) not in sys.path:
    sys.path.insert(0, str(SURFACE_DIR))
import surface_extraction as se


In [ ]:
# Input folder and filename ending. Subfolders are included.
INPUT_DIR = '/home/khan365/Point-M2AE-main/LS_Slopes_reconstruction_Multiple_iterations/results/pointm2ae_recon_90M/point_clouds/'
FILE_SUFFIX = '_first_predictions.las'  # e.g. '_representative.las', '.las', or '.laz'

# Choose ONE: 'voxel', 'xy_grid', 'mls', 'local_plane'.
SURFACE_TYPE = 'mls'

# Edit the parameters for your selected type; the others are ignored.
VOXEL = dict(size=0.5)
XY_GRID = dict(
    size=1,
    statistic='median',  # 'mean', 'median', 'minimum', 'maximum', 'percentile'
    minimum_points=1,
    percentile=10.0,   # 0–100; used only when statistic='percentile'.
)
MLS = dict(
    neighbors=64,
    radius=3.0,
    degree=2,
    bandwidth=1.0,
    minimum_neighbors=10,
)
LOCAL_PLANE = dict(
    neighbors=16,
    radius=3.0,
    bandwidth=1.0,
    minimum_neighbors=10,
)

OUTPUT_DIR = SURFACE_DIR / 'results' / (
    'surface_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    + '_' + uuid.uuid4().hex[:8]
)


## Reconstruct and save

Run the following cell to apply the selected method to every matching file.
Output subfolders mirror the input subfolders. Each input produces
`OUTPUT_DIR/<input subfolders>/<original stem>_surface.las` and a matching `.npz` archive.
For example, `LAM_1/SU_27077_first_predictions.las` becomes
`LAM_1/SU_27077_first_predictions_surface.las` inside the output folder.
The LAS preserves the input coordinate system and coordinate precision.
`summary.csv` records point counts, fallback counts and processing time;
`manifest.json` records the parameters, and `failures.json` records files that failed.
Use a new output folder for each run; rerunning the settings cell generates one.


In [41]:
import importlib

# Refresh helper code when this notebook uses an existing kernel.
se = importlib.reload(se)

INPUT_DIR = Path(INPUT_DIR).expanduser()
if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f'Input folder not found: {INPUT_DIR}')
if (not isinstance(FILE_SUFFIX, str) or not FILE_SUFFIX
        or any(char in FILE_SUFFIX for char in '*?[]/\\')
        or not FILE_SUFFIX.lower().endswith(('.las', '.laz'))):
    raise ValueError('FILE_SUFFIX must be a filename ending such as _first_predictions.las or .laz')

if SURFACE_TYPE == 'voxel':
    method = 'voxel'
    config = se.SurfaceConfig(voxel_size=VOXEL['size'])
elif SURFACE_TYPE == 'xy_grid':
    if XY_GRID['statistic'] not in ('mean', 'median', 'minimum', 'maximum', 'percentile'):
        raise ValueError('Choose mean, median, minimum, maximum or percentile for XY_GRID statistic')
    method = 'xy_' + XY_GRID['statistic']
    config = se.SurfaceConfig(xy_grid_size=XY_GRID['size'],
                              xy_minimum_points=XY_GRID['minimum_points'],
                              percentile=XY_GRID['percentile'])
elif SURFACE_TYPE == 'mls':
    method = 'mls'
    config = se.SurfaceConfig(**{f'mls_{key}': value for key, value in MLS.items()})
elif SURFACE_TYPE == 'local_plane':
    method = 'local_plane'
    config = se.SurfaceConfig(**{f'local_{key}': value for key, value in LOCAL_PLANE.items()})
else:
    raise ValueError('SURFACE_TYPE must be voxel, xy_grid, mls or local_plane')

manifest = se.run_surface_pipeline(
    INPUT_DIR, OUTPUT_DIR, config,
    source_glob='*' + FILE_SUFFIX,
    methods=[method],
    mirror_source=True,
    plot_units=0,
)
print('Output folder:', manifest['output_dir'])
print('Method:', method)
print('Files:', manifest['counts'])
if manifest['counts']['failed']:
    print('See failures.json in the output folder for failed files.')


1/1251: {'selected': 1251, 'processed': 1, 'failed': 0}; 0.0s
62/1251: {'selected': 1251, 'processed': 62, 'failed': 0}; 10.7s
124/1251: {'selected': 1251, 'processed': 124, 'failed': 0}; 21.1s
186/1251: {'selected': 1251, 'processed': 186, 'failed': 0}; 33.3s
248/1251: {'selected': 1251, 'processed': 248, 'failed': 0}; 46.1s
310/1251: {'selected': 1251, 'processed': 310, 'failed': 0}; 58.6s
372/1251: {'selected': 1251, 'processed': 372, 'failed': 0}; 70.0s
434/1251: {'selected': 1251, 'processed': 434, 'failed': 0}; 82.6s
496/1251: {'selected': 1251, 'processed': 496, 'failed': 0}; 98.6s
558/1251: {'selected': 1251, 'processed': 558, 'failed': 0}; 114.4s
620/1251: {'selected': 1251, 'processed': 620, 'failed': 0}; 127.4s
682/1251: {'selected': 1251, 'processed': 682, 'failed': 0}; 139.6s
744/1251: {'selected': 1251, 'processed': 744, 'failed': 0}; 153.9s
806/1251: {'selected': 1251, 'processed': 806, 'failed': 0}; 168.6s
868/1251: {'selected': 1251, 'processed': 868, 'failed': 0}; 182